# IRS-to-LLC Map Verification

Build and inspect the **`IRS2LLC_map`** for every supported IRS form (Form 1065, Schedule K-1, Form 4562).

The map is the `create-once / use-N-times` table that drives `FILL.pdf` generation. This notebook's job is to **verify** the map before we wire `irs.*` code to consume it.

Pipeline
```
irsFormObj  →  stmt.mapIRS2LLC._mapIRS2LLC(form)  →  list[formLineDict]
                                                     (one per AcroForm fid)
```

Each `formLineDict` answers

| field | meaning |
|---|---|
| `fid` | PDF AcroForm field id (`f1`…`fN`) |
| `loc_dataObjectClassName` | which LLC data object provisions this cell (`None` if nobody) |
| `loc_tbl_id` | table id within that data object |
| `loc_rowNm` | row name within that table |
| `loc_colNm` | column name within that row |
| `value` | resolved value (`None` if no claim / no value) |

Data-object priority (authoritative CPA / IRS order)
`IS → BS → Customers → Owners → LLC (profile) → GeneralLedger`

The first object to claim an `fid` wins; later objects only fill unclaimed fids.

## Setup

In [4]:
import sys, os
# Notebook runs from .../pages/AccountingData/Notebooks
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from IPython.display import display, Markdown
import pandas as pd
from ledger.LLC        import LLC
from irs.Form1065      import Form1065
from irs.Sch_K1        import Sch_K1
from irs.Form4562      import Form4562
from irs.mapIRS2LLC   import _mapIRS2LLC, formLineDict

pd.set_option('display.max_rows', 250)
pd.set_option('display.width',    220)
pd.set_option('display.max_colwidth', 40)

llc = LLC('WBGroupLLC')
# Pre-initialize bank so stmtFinancialReport.taxAggregates works everywhere.
llc._Bank()
print(f'LLC ready: {llc.objName}  year={llc.yr}')

## Helper: build + summarise the IRS2LLC map

def build_map(form):
    '''Return (DataFrame, summary dict) for the given IRS form instance.'''
    rows = _mapIRS2LLC(form)
    df   = pd.DataFrame(rows, columns=list(formLineDict.__dataclass_fields__.keys()))

    # Summary
    total    = len(df)
    claimed  = int(df['loc_dataObjectClassName'].notna().sum())
    unfilled = total - claimed
    by_obj   = df['loc_dataObjectClassName'].fillna('<unclaimed>').value_counts().to_dict()
    with_val = int(df['value'].notna().sum())

    summary = {
        'form':        getattr(form, 'oID', form.__class__.__name__),
        'total_fids':  total,
        'claimed':     claimed,
        'unclaimed':   unfilled,
        'with_value':  with_val,
        'by_dataObj':  by_obj,
    }
    return df, summary

def show_claimed(df, n=30):
    '''Return only the rows a data object has claimed.'''
    claimed = df[df['loc_dataObjectClassName'].notna()].copy()
    return claimed.head(n)

#
# -------- set up all financial objects
#
from util.utilEditSession import utilEditSession
from ui.llcMgmt import llcMgmt

ui = llcMgmt(utilEditSession(llcName='WBGroupLLC'))
oDict = ui._build_objects()

# --- display financial objects
display(Markdown(f'\n<h3>Financial LLC Object (oDict)'))
display(Markdown(f'{oDict.keys()}'))



LLC ready: WBGroupLLC  year=2025
⚠️  Warning Loan Failed: wkAP working file not found. Correct and restart, /tmp/llcPayables_WBGroupLLC_temp.json, loadOpt:False
⚠️  Warning Loan Failed: wkAR working file not found. Correct and restart, /tmp/llcReceivables_WBGroupLLC_temp.json, loadOpt:False
99 eSession bind objects: {'wkAssets': '/tmp/llcAssets_WBGroupLLC_temp.json', 'wkExpRev': '/tmp/llcExpRev_WBGroupLLC_temp.json', 'wkAP': '/tmp/llcPayables_WBGroupLLC_temp.json', 'wkAR': '/tmp/llcReceivables_WBGroupLLC_temp.json'}



<h3>Financial LLC Object (oDict)

dict_keys(['llcAssets', 'llcExpRev', 'llcPayables', 'llcReceivables', 'stmtGeneralLedger', 'stmtBalanceSheet', 'stmtIncomeStmt', 'stmtOwnerEquity', 'llcBank', 'stmtPropertyEquity', 'llcForm1065', 'llcFormK1', 'llcFormSchedL', 'llcFormSchedM1', 'llcFormSchedM2', 'llcForm1065SchBPg2', 'llcForm1065SchBPg3', 'llcForm1065SchBPg4', 'llcForm1065SchKPg5', 'llcForm1065Pg6'])

In [25]:
# Build oTypeDict : map obj -> type [ledger, stmt, irs]
lList = os.listdir('ledger')
sList = os.listdir('stmt')
oTypeDict = {}
for o in oDict.keys():
    pyNm = f'{o}.py'
    
    if pyNm in lList : 
        oTypeDict[o] = 'ledger' 
        continue
    elif pyNm in sList : 
        oTypeDict[o] = 'stmt'
        continue
    elif 'llcForm' in o :
        oTypeDict[o] = 'irs'
        continue
    oTypeDict[o] = None
#oTypeDict 

In [44]:
for o in oDict:
    fObj = oDict[o]
    fNm = str(fObj)
    
    print(fNm)

In [53]:
form1065 = Form1065(llc=llc)
form_k1 = Sch_K1(llc=llc)
form_4562 = Form4562(llc=llc)

for o in oDict:
    if oTypeDict[o] != 'stmt' : continue

    fObj = oDict[o]
    fNm = str(fObj).replace('<','').split()[0]
    display(Markdown(h))
    for tObj in [form1065, form_k1, form_4562]:
        try:
            df = fObj._to_IRS(tObj)
        except:
            df = None
        display(Markdown(f"<h4> --- {tObj.oID}._to_IRS"))
        display(df)
    

<h2> stmtPropertyEquity: Financial Obj : ui.stmtPropertyEquity.stmtPropertyEquity

<h4> --- Form1065._to_IRS

None

<h4> --- Sch_K1._to_IRS

None

...

None

<h4> --- Sch_K1._to_IRS

None

<h4> --- Form4562._to_IRS

None

## Form 1065 (Pages 1–6)

In [3]:
form1065 = Form1065(llc=llc)
df_1065, sum_1065 = build_map(form1065)
sum_1065

_mapIRS2LLC: stmtIncomeStmt._to_IRS failed: No module named 'mapIRS2LLC'
_mapIRS2LLC: stmtBalanceSheet._to_IRS failed: No module named 'mapIRS2LLC'
_mapIRS2LLC: llcOwners._to_IRS failed: No module named 'mapIRS2LLC'
_mapIRS2LLC: LLC._to_IRS failed: No module named 'mapIRS2LLC'


{'form': 'Form1065',
 'total_fids': 500,
 'claimed': 0,
 'unclaimed': 500,
 'with_value': 0,
 'by_dataObj': {'<unclaimed>': 500}}

In [34]:
# Full table (every fid, one row) — sorted by numeric fid.
df_1065

,fid,loc_dataObjectClassName,loc_tbl_id,loc_rowNm,loc_colNm,value
0,f1,None,None,None,None,None
1,f2,None,None,None,None,None
2,f3,None,None,None,None,None
3,f4,None,None,None,None,None
4,f5,None,None,None,None,None
...,...,...,...,...,...,...
495,f496,None,None,None,None,None
496,f497,None,None,None,None,None
497,f498,None,None,None,None,None
498,f499,None,None,None,None,None


In [35]:
# Only the fids claimed by a data object.
show_claimed(df_1065, n=100)

,fid,loc_dataObjectClassName,loc_tbl_id,loc_rowNm,loc_colNm,value


## Schedule K-1 (Form 1065)

K-1 is rendered once per partner. The Owners column ordering in `_to_IRS` follows the `llcOwners` list: `owner[0]` is the first partner, `owner[1]` the second, etc.  The consumer (FILL writer) picks the appropriate `owner[i]` row when generating each partner's PDF.

In [36]:
form_k1 = Sch_K1(llc=llc)
df_k1, sum_k1 = build_map(form_k1)
sum_k1

irsForm Entry: oID:Sch_K1, verbose:False
Acct.Rev List: 12, revList:7
Acct.Rev List: 12, revList:7


{'form': 'Sch_K1',
 'total_fids': 137,
 'claimed': 0,
 'unclaimed': 137,
 'with_value': 0,
 'by_dataObj': {'<unclaimed>': 137}}

In [37]:
df_k1

,fid,loc_dataObjectClassName,loc_tbl_id,loc_rowNm,loc_colNm,value
0,f1,None,None,None,None,None
1,f2,None,None,None,None,None
2,f3,None,None,None,None,None
3,f4,None,None,None,None,None
4,f5,None,None,None,None,None
5,f6,None,None,None,None,None
6,f7,None,None,None,None,None
7,f8,None,None,None,None,None
8,f9,None,None,None,None,None
9,f10,None,None,None,None,None


In [38]:
show_claimed(df_k1, n=100)

,fid,loc_dataObjectClassName,loc_tbl_id,loc_rowNm,loc_colNm,value


## Form 4562 (Depreciation & Amortization)

In [39]:
form_4562 = Form4562(llc=llc)
df_4562, sum_4562 = build_map(form_4562)
sum_4562

irsForm Entry: oID:Form4562, verbose:False
Acct.Rev List: 12, revList:7
Acct.Rev List: 12, revList:7
Acct.Rev List: 12, revList:7
Acct.Rev List: 12, revList:7


{'form': 'Form4562',
 'total_fids': 323,
 'claimed': 0,
 'unclaimed': 323,
 'with_value': 0,
 'by_dataObj': {'<unclaimed>': 323}}

In [40]:
df_4562

,fid,loc_dataObjectClassName,loc_tbl_id,loc_rowNm,loc_colNm,value
0,f1,None,None,None,None,None
1,f2,None,None,None,None,None
2,f3,None,None,None,None,None
3,f4,None,None,None,None,None
4,f5,None,None,None,None,None
...,...,...,...,...,...,...
318,f319,None,None,None,None,None
319,f320,None,None,None,None,None
320,f321,None,None,None,None,None
321,f322,None,None,None,None,None


In [41]:
show_claimed(df_4562, n=100)

,fid,loc_dataObjectClassName,loc_tbl_id,loc_rowNm,loc_colNm,value


## Display per financial tax data using `to_IRS`

In [42]:
from stmt.stmtIncomeStmt import stmtIncomeStmt
from stmt.stmtBalanceSheet import stmtBalanceSheet

display(Markdown("<h2>LLC._to_IRS"))
display(llc._to_IRS(form1065))

display(Markdown("\n<h2>stmtIncomeStmt._to_IRS"))
display(stmtIncomeStmt(llc)._to_IRS(form1065))

display(Markdown("\n<h2>stmtIncomeStmt._to_IRS"))
display(stmtBalanceSheet(llc)._to_IRS(form1065))


<h2>LLC._to_IRS

[]


<h2>stmtIncomeStmt._to_IRS

Acct.Rev List: 12, revList:7
Acct.Rev List: 12, revList:7


[]


<h2>stmtIncomeStmt._to_IRS

Acct.Rev List: 12, revList:7
Acct.Rev List: 12, revList:7


[]

In [ ]:
from ui.stmtIncomeStmt import stmtIncomeStmt
from ui.stmtBalanceSheet import stmtBalanceSheet



## Cross-form check: coverage by data object

How many cells each data object provisions across the full IRS form set.

In [12]:
import collections
coverage = collections.Counter()
for name, df in [('Form1065', df_1065), ('Sch_K1', df_k1), ('Form4562', df_4562)]:
    for obj, n in df['loc_dataObjectClassName'].fillna('<unclaimed>').value_counts().items():
        coverage[(name, obj)] = int(n)
cov_df = pd.DataFrame(
    [(f, o, n) for (f, o), n in coverage.items()],
    columns=['form', 'loc_dataObjectClassName', 'count']
).sort_values(['form', 'loc_dataObjectClassName']).reset_index(drop=True)
cov_df

,form,loc_dataObjectClassName,count
0,Form1065,<unclaimed>,500
1,Form4562,<unclaimed>,323
2,Sch_K1,<unclaimed>,137


## CPA review queue: unclaimed fids

These are the cells no data object provisions yet.  They are either:

1. Legitimately CPA-review (statutory limits, elections, Part-V listed property)
2. Checkbox / signature / date-signed cells the partnership itself must fill
3. Gaps we still need to close in a future `_IRS_BINDINGS` expansion

In [13]:
def unclaimed(df):
    return df[df['loc_dataObjectClassName'].isna()].copy()

print(f'Form1065 unclaimed: {len(unclaimed(df_1065))}')
print(f'Sch_K1   unclaimed: {len(unclaimed(df_k1))}')
print(f'Form4562 unclaimed: {len(unclaimed(df_4562))}')

Form1065 unclaimed: 500
Sch_K1   unclaimed: 137
Form4562 unclaimed: 323


---
**Verification status**

- `stmt.mapIRS2LLC._mapIRS2LLC(form)` returns one `formLineDict` per AcroForm fid.
- Data objects declare their IRS bindings internally via `_to_IRS(formObj)` — `_mapIRS2LLC` has no per-form knowledge.
- Data-object priority: `stmtIncomeStmt` → `stmtBalanceSheet` → `llcCustomers` → `llcOwners` → `LLC` (profile) → `stmtGeneralLedger`.
- The map is create-once / use-N-times — downstream `FILL.pdf` writers consume this table without rebuilding it.

Next step (out of scope for this notebook): wire `irs.Form1065.buildFillDict()` / `irs.Sch_K1.buildFillDict()` / `irs.Form4562.buildFillDict()` to use `_mapIRS2LLC` instead of the legacy `_FILL_MAP` resolution path.